# NetlogRAG fine-tuning v3

Reproducibilan Colab orkestrator za pripremu podataka, QLoRA fine-tuning i evaluaciju. Pokrenite svaki model u zasebnoj GPU sesiji. Ovaj notebook ne zamjenjuje skripte: one su verzionirani izvor istine.

Za nenadzirano izvođenje notebook zapisuje status ili pogrešku na Google Drive te, prema postavkama ispod, automatski oslobađa Colab runtime.

## 1. Repozitorij i ovisnosti
Odaberite GPU runtime. Grana je namjerno zaključana na `codex/thesis-revision`; `main` se ne mijenja.

In [ ]:
import os
import pathlib
import subprocess
import sys

REPO = pathlib.Path("/content/RazvojIT-rijesenja")
if not REPO.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "codex/thesis-revision",
            "https://github.com/lovro52/RazvojIT-rijesenja.git",
            str(REPO),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
os.chdir(REPO)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "experiments/requirements-colab.txt"],
    check=True,
)

## 2. Putanje i trajna pohrana
Postavite direktorije s originalnim CICIDS2017 CSV datotekama. CSE-CIC-IDS2018 je opcionalan vanjski test i ne ulazi u trening.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

CICIDS2017_DIR = pathlib.Path("/content/drive/MyDrive/NetlogRAG/data/CICIDS2017")
CICIDS2018_DIR = pathlib.Path("/content/drive/MyDrive/NetlogRAG/data/CSE-CIC-IDS2018")
DATA_DIR = pathlib.Path("/content/drive/MyDrive/NetlogRAG/prepared-v3")
ARTIFACT_DIR = pathlib.Path("/content/drive/MyDrive/NetlogRAG/artifacts-v3")
REPORT_DIR = pathlib.Path("/content/drive/MyDrive/NetlogRAG/reports-v3")
assert CICIDS2017_DIR.exists(), f"Nije pronađen {CICIDS2017_DIR}"
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

## 3.1. Sigurno nenadzirano izvođenje

`AUTO_DISCONNECT` oslobađa runtime nakon uspješnog treninga i evaluacije. `DISCONNECT_ON_ERROR` prije odspajanja sprema zapis pogreške na Drive. Tijekom interaktivnog ispravljanja pogrešaka možete privremeno postaviti `DISCONNECT_ON_ERROR = False`.

In [ ]:
import datetime as dt
import json
import time
import traceback
from google.colab import runtime

AUTO_DISCONNECT = True
DISCONNECT_ON_ERROR = True

def _write_colab_status(status, *, details=None, files=None):
    model = globals().get("MODEL", "setup")
    payload = {
        "status": status,
        "model": model,
        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "files": files or [],
    }
    (REPORT_DIR / f"{model}-colab-session.json").write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    if details:
        (REPORT_DIR / f"{model}-colab-error.txt").write_text(
            details,
            encoding="utf-8",
        )

def _disconnect_after_failed_cell(result):
    error = result.error_in_exec or result.error_before_exec
    if error is None or not DISCONNECT_ON_ERROR:
        return
    details = "".join(
        traceback.format_exception(type(error), error, error.__traceback__)
    )
    try:
        _write_colab_status("failed", details=details)
        drive.flush_and_unmount()
        time.sleep(5)
    finally:
        runtime.unassign()

_ipython = get_ipython()
_previous_handler = globals().get("_AUTO_DISCONNECT_HANDLER")
if _previous_handler is not None:
    try:
        _ipython.events.unregister("post_run_cell", _previous_handler)
    except ValueError:
        pass
_AUTO_DISCONNECT_HANDLER = _disconnect_after_failed_cell
_ipython.events.register("post_run_cell", _AUTO_DISCONNECT_HANDLER)

print(
    f"Automatsko odspajanje: {AUTO_DISCONNECT}; "
    f"odspajanje nakon pogreške: {DISCONNECT_ON_ERROR}"
)

## 3. Priprema zaključanih splitova
Fingerprint grupiranje sprječava da identični vektori značajki prijeđu između treninga i testa. Izvještaj sprema SHA-256 izvornih datoteka.

In [ ]:
command = [
    sys.executable,
    "-m",
    "experiments.prepare_dataset",
    "--input-dir",
    str(CICIDS2017_DIR),
    "--output-dir",
    str(DATA_DIR),
    "--max-per-class",
    "5000",
    "--seed",
    "42",
]
if CICIDS2018_DIR.exists():
    command += ["--external-dir", str(CICIDS2018_DIR)]
subprocess.run(command, check=True)

## 4. Odabir jednog modela i QLoRA trening
Dopuštene vrijednosti su `qwen3-1.7b`, `smollm3-3b` i `phi4-mini`. Nakon jednog modela oslobodite runtime i ponovite notebook za sljedeći. Ako je isti trening ranije prekinut, postavite `RESUME_FROM_CHECKPOINT = True` kako bi se nastavio iz posljednjeg spremljenog checkpointa.

In [ ]:
MODEL = "qwen3-1.7b"
RESUME_FROM_CHECKPOINT = False

train_command = [
    sys.executable,
    "-m",
    "experiments.finetune",
    "--model",
    MODEL,
    "--data-dir",
    str(DATA_DIR),
    "--output-dir",
    str(ARTIFACT_DIR),
    "--epochs",
    "2",
    "--seed",
    "42",
]
if RESUME_FROM_CHECKPOINT:
    train_command.append("--resume")

subprocess.run(train_command, check=True)

## 5. Usporediva base i fine-tuned evaluacija
Obje varijante koriste isti test. Ne podešavajte hiperparametre prema tim rezultatima.

In [ ]:
test_file = DATA_DIR / "test.jsonl"
adapter = ARTIFACT_DIR / MODEL / "adapter"
for variant in ("base", "fine_tuned"):
    command = [
        sys.executable,
        "-m",
        "experiments.evaluate",
        "--model",
        MODEL,
        "--variant",
        variant,
        "--dataset",
        str(test_file),
        "--output",
        str(REPORT_DIR / f"{MODEL}-{variant}-test.json"),
        "--seed",
        "42",
    ]
    if variant == "fine_tuned":
        command += ["--adapter", str(adapter)]
    subprocess.run(command, check=True)

## 6. Vanjski test (ako postoji)
CSE-CIC-IDS2018 služi procjeni promjene distribucije. Zbog različitih scenarija i oznaka rezultat se izvještava zasebno.

In [ ]:
external = DATA_DIR / "external_test.jsonl"
if external.exists():
    subprocess.run(
        [
            sys.executable,
            "-m",
            "experiments.evaluate",
            "--model",
            MODEL,
            "--variant",
            "fine_tuned",
            "--adapter",
            str(adapter),
            "--dataset",
            str(external),
            "--output",
            str(REPORT_DIR / f"{MODEL}-fine_tuned-external.json"),
            "--seed",
            "42",
        ],
        check=True,
    )
else:
    print("Vanjski test nije pripremljen.")

## 7. Provjera rezultata i automatsko odspajanje

Ova ćelija provjerava obvezne izlazne datoteke, zapisuje uspješan završetak na Drive, sinkronizira Drive i tek tada oslobađa Colab runtime. Mora ostati posljednja izvršna ćelija.

In [ ]:
expected_files = [
    DATA_DIR / "dataset_report.json",
    ARTIFACT_DIR / MODEL / "adapter" / "adapter_config.json",
    ARTIFACT_DIR / MODEL / "training_report.json",
    REPORT_DIR / f"{MODEL}-base-test.json",
    REPORT_DIR / f"{MODEL}-base-test_predictions.jsonl",
    REPORT_DIR / f"{MODEL}-fine_tuned-test.json",
    REPORT_DIR / f"{MODEL}-fine_tuned-test_predictions.jsonl",
]
if external.exists():
    expected_files.extend(
        [
            REPORT_DIR / f"{MODEL}-fine_tuned-external.json",
            REPORT_DIR / f"{MODEL}-fine_tuned-external_predictions.jsonl",
        ]
    )

missing_files = [str(path) for path in expected_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Ne odspajam runtime kao uspješno dovršen jer nedostaju datoteke:\n"
        + "\n".join(missing_files)
    )

_write_colab_status(
    "completed",
    files=[str(path) for path in expected_files],
)
print("Svi obvezni rezultati spremljeni su na Google Drive.")

if AUTO_DISCONNECT:
    try:
        _ipython.events.unregister("post_run_cell", _AUTO_DISCONNECT_HANDLER)
    except ValueError:
        pass
    drive.flush_and_unmount()
    time.sleep(5)
    runtime.unassign()
else:
    print("AUTO_DISCONNECT je isključen; runtime ostaje povezan.")

## 8. Sljedeći korak: kvantizacija i lokalna brzina
Nakon treninga slijedite `experiments/README.md`: spojite adapter, izvezite GGUF Q4_K_M i mjerite stvarni Ollama runtime s `experiments.evaluate_ollama`. Nemojte nazvati sustav real-time prije nego što definirate ciljani broj tokova u sekundi i usporedite ga s izmjerenom propusnošću.